### 1 — Import Libraries

In [ ]:
from pyspark.sql import SparkSession
import re, os, uuid
from pyspark.sql.utils import AnalysisException
from pyspark.sql import Row
import json
from pyspark.sql.functions import *
import pyspark.sql.functions as Fun
from pyspark.sql.types import *
from pyspark.sql.window import Window
from datetime import datetime, timedelta


### 2 — Lakehouse & Container

In [ ]:
# ── Lakehouse identity ─────────────────────────────────────────────────────────
Lakehouse_Name         = ""
Lakehouse_Storage_Path = ""
Container_Name         = ""
Industry_Vertical      = ""

print(f"Lakehouse : {Lakehouse_Name}")
print(f"Container : {Container_Name}")


### 3 — Layer Paths  (Files/ used for backup only)

In [ ]:
# ── Derived base path ──────────────────────────────────────────────────────────
Lakehouse_Folder_Storage_Path = f"{Lakehouse_Storage_Path}/{Container_Name}"

# ── Layer folder names ─────────────────────────────────────────────────────────
Landing_Folder_Name = "RAW_LAYER"
Bronze_Folder_Name  = "BRONZE_LAYER"
Silver_Folder_Name  = "SILVER_LAYER"
Gold_Folder_Name    = "GOLD_LAYER"
Error_Folder_Name   = "ERROR_LOG_LAYER"

# ── Full layer paths (Files/ — used ONLY for backup) ──────────────────────────
Landing_Folder_Path  = f"{Lakehouse_Folder_Storage_Path}/{Landing_Folder_Name}"
Bronze_Backup_Path   = f"{Lakehouse_Folder_Storage_Path}/{Bronze_Folder_Name}"
Silver_Backup_Path   = f"{Lakehouse_Folder_Storage_Path}/{Silver_Folder_Name}"
Gold_Backup_Path     = f"{Lakehouse_Folder_Storage_Path}/{Gold_Folder_Name}"
Error_Table_Location = f"{Lakehouse_Folder_Storage_Path}/{Error_Folder_Name}"

print(f"Landing  : {Landing_Folder_Path}")
print(f"Error    : {Error_Table_Location}")


### 4 — Schema Names & Table Constants

In [ ]:
# ── Managed Lakehouse schema names (dot-notation tables live here) ─────────────
Bronze_Schema = "bronze_legal"
Silver_Schema = "silver_legal"
Gold_Schema   = "gold_legal"
Error_Schema  = "error_legal"

# ── Table name constants ───────────────────────────────────────────────────────
CENTRAL_ERROR_LOG_TABLE = "legal_central_error_logs"

# ── Ensure schemas exist (idempotent) ─────────────────────────────────────────
for _schema in [Bronze_Schema, Silver_Schema, Gold_Schema, Error_Schema]:
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {_schema}")
    print(f"Schema ready : {_schema}")


### 5 — Backup Toggle & Retention

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  BACKUP TOGGLE
#  Set ENABLE_FILES_BACKUP = True to also write a point-in-time Delta snapshot
#  of every table to Files/  after the main pipeline completes.
#  All other code is unaffected when this is False.
# ══════════════════════════════════════════════════════════════════════════════
ENABLE_FILES_BACKUP    = False   # ← flip to True when a backup run is needed
BACKUP_RETENTION_DAYS  = 7       # ← dated backup folders older than this are pruned


### 6 — Backup / Restore Helper Functions

In [ ]:
# ── Shared backup helper (used by Raw-to-Bronze, Bronze-to-Silver, Silver-to-Gold) ──

def backup_tables_to_files(tables: list, backup_root: str, layer_label: str):
    """
    Writes managed Lakehouse tables as Delta files under
    backup_root/BACKUP/YYYY-MM-DD/<table_name>/.

    Parameters
    ----------
    tables      : list of (table_name, schema_name) tuples
    backup_root : base Files/ path for this layer  (e.g. Bronze_Backup_Path)
    layer_label : string label used in print output (e.g. "BRONZE_BACKUP")
    """
    backup_date = datetime.now().strftime("%Y-%m-%d")
    success, failed = [], []

    for table_name, schema_name in tables:
        dest = f"{backup_root}/BACKUP/{backup_date}/{table_name}"
        try:
            df = spark.read.table(f"{schema_name}.{table_name}")
            df.write.mode("overwrite").format("delta").option("path", dest).save()
            print(f"[BACKUP ✓] {schema_name}.{table_name}  →  {dest}")
            success.append(table_name)
        except Exception as _be:
            print(f"[BACKUP ✗] {schema_name}.{table_name}  |  {_be}")
            failed.append((table_name, str(_be)))

    print(f"\n[BACKUP SUMMARY] layer={layer_label} | "
          f"success={len(success)} | failed={len(failed)} | date={backup_date}")
    return success, failed


def restore_from_backup(table_name: str, schema_name: str,
                        backup_root: str, backup_date: str):
    """
    Restores a managed table from a dated Files/ backup snapshot.
    backup_date format: "YYYY-MM-DD"

    Usage:
        restore_from_backup("raw_lawyer_profile", Silver_Schema,
                            Silver_Backup_Path, "2026-06-03")
    """
    src = f"{backup_root}/BACKUP/{backup_date}/{table_name}"
    df  = spark.read.format("delta").load(src)
    df.write.mode("overwrite").format("delta") \
        .option("mergeSchema", "true") \
        .saveAsTable(f"{schema_name}.{table_name}")
    print(f"[RESTORE ✓] {schema_name}.{table_name} restored from {src}")


def _prune_old_backups(backup_root: str, retention_days: int):
    """Deletes backup date-folders older than retention_days. Non-fatal."""
    try:
        cutoff   = datetime.now() - timedelta(days=retention_days)
        bk_path  = f"{backup_root}/BACKUP"
        all_dirs = mssparkutils.fs.ls(bk_path)
        for d in all_dirs:
            try:
                folder_date = datetime.strptime(d.name, "%Y-%m-%d")
                if folder_date < cutoff:
                    mssparkutils.fs.delete(d.path, recurse=True)
                    print(f"[BACKUP PRUNED] {d.path}")
            except ValueError:
                pass   # skip non-date folders
    except Exception as _pe:
        print(f"[BACKUP PRUNE WARNING] {_pe}")


print("Backup helpers registered: backup_tables_to_files | restore_from_backup | _prune_old_backups")
